# 06 — Visualising Attention

Now that we have a trained model, let's look inside and see what the attention heads actually learned. We'll visualise per-layer, per-head attention patterns, compute attention rollout, and explore how different tokens attend to each other.

## Table of Contents
1. [Load the Trained Model](#1-load-the-trained-model)
2. [Per-Layer, Per-Head Heatmaps](#2-per-layer-per-head-heatmaps)
3. [Average Attention Per Head](#3-average-attention-per-head)
4. [Attention Rollout](#4-attention-rollout)
5. [Interactive: Visualise Your Own Text](#5-interactive-visualise-your-own-text)
6. [Interpreting Attention Patterns](#6-interpreting-attention-patterns)

---

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tiktoken
import sys, os

sys.path.insert(0, os.path.abspath('.'))
from utils.model import GPTModel
from utils.visualisation import (
    plot_attention_heatmap,
    plot_multi_head_attention,
    plot_attention_rollout,
    compute_attention_rollout
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load the Trained Model

In [ ]:
# ============================================================
# Load checkpoint from notebook 05
# ============================================================

checkpoint_path = "checkpoints/model.pt"

if not os.path.exists(checkpoint_path):
    print("ERROR: No checkpoint found!")
    print("Please run notebook 05_training.ipynb first to train and save a model.")
else:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    config = checkpoint["config"]
    
    # Recreate model with the same config
    model = GPTModel(config)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()  # Set to evaluation mode (no dropout)
    
    # Tokenizer
    enc = tiktoken.get_encoding("gpt2")
    
    print(f"Model loaded from: {checkpoint_path}")
    print(f"Config: d_model={config['d_model']}, n_heads={config['n_heads']}, n_layers={config['n_layers']}")
    print(f"Training loss at save: {checkpoint.get('final_train_loss', 'N/A'):.4f}")
    print(f"Parameters: {model.count_parameters():,}")

In [ ]:
# ============================================================
# Helper: Run a sentence through the model and extract attention
# ============================================================

def get_attention_for_text(model, text, enc):
    """
    Run text through the model and return attention weights + token strings.
    
    Returns:
        attn_weights: list of (n_heads, seq, seq) tensors, one per layer
        tokens: list of token strings
    """
    # Tokenise
    token_ids = enc.encode(text)
    tokens = [enc.decode([t]) for t in token_ids]
    
    # Run through model
    input_ids = torch.tensor([token_ids], dtype=torch.long)  # (1, seq)
    with torch.no_grad():
        _ = model(input_ids)  # Forward pass (populates attention caches)
    
    # Extract attention weights from each layer
    attn_weights = model.get_attention_weights()
    # Each element: (1, n_heads, seq, seq) → remove batch dim
    attn_weights = [w.squeeze(0) for w in attn_weights]  # (n_heads, seq, seq)
    
    return attn_weights, tokens

# Test with a sample sentence
text = "Engineering is the application of scientific principles"
attn_weights, tokens = get_attention_for_text(model, text, enc)

print(f"Input text: \"{text}\"")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"Number of layers: {len(attn_weights)}")
print(f"Attention shape per layer: {attn_weights[0].shape}  — (n_heads, seq, seq)")

## 2. Per-Layer, Per-Head Heatmaps

Each layer has multiple attention heads, and each head learns to focus on different things. Let's visualise them all.

In [ ]:
# ============================================================
# Visualise ALL attention heads across ALL layers
# ============================================================

n_layers = len(attn_weights)
n_heads = attn_weights[0].shape[0]

for layer_idx in range(n_layers):
    print(f"\n{'='*60}")
    print(f"LAYER {layer_idx}")
    print(f"{'='*60}")
    
    plot_multi_head_attention(
        attn_weights[layer_idx],  # (n_heads, seq, seq)
        tokens=tokens,
        title=f"Layer {layer_idx} — All {n_heads} Heads",
        annot=len(tokens) <= 10  # annotate if sequence is short enough
    )

In [ ]:
# ============================================================
# Zoom into a single head for detailed inspection
# ============================================================

layer_idx = 0
head_idx = 0

plot_attention_heatmap(
    attn_weights[layer_idx][head_idx],
    tokens=tokens,
    title=f"Layer {layer_idx}, Head {head_idx} — Detailed View",
    figsize=(9, 8)
)

# Print the attention distribution for each query
print(f"\nLayer {layer_idx}, Head {head_idx} — Attention distribution:")
print("-"*60)
weights = attn_weights[layer_idx][head_idx].numpy()
for i in range(len(tokens)):
    top_k = 3
    sorted_idx = np.argsort(weights[i])[::-1][:top_k]
    print(f"  '{tokens[i]:>15}' attends to:", end="")
    for j in sorted_idx:
        if weights[i][j] > 0.01:
            print(f"  '{tokens[j]}' ({weights[i][j]:.2f})", end="")
    print()

## 3. Average Attention Per Head

Averaging across positions shows each head's general tendency.

In [ ]:
# ============================================================
# Average attention per head: What's each head's "style"?
# ============================================================

fig, axes = plt.subplots(n_layers, n_heads, figsize=(4 * n_heads, 4 * n_layers))
if n_layers == 1:
    axes = [axes]

for layer_idx in range(n_layers):
    for head_idx in range(n_heads):
        ax = axes[layer_idx][head_idx] if n_layers > 1 else axes[head_idx]
        
        w = attn_weights[layer_idx][head_idx].numpy()
        seq_len = w.shape[0]
        
        # For each query position, where does this head tend to look?
        # Create a "relative position" heatmap
        # x-axis: relative position (how far back does it attend?)
        # y-axis: query position
        
        sns.heatmap(w, ax=ax, cmap='Blues', vmin=0, cbar=False)
        ax.set_title(f'L{layer_idx} H{head_idx}', fontsize=10, fontweight='bold')
        if head_idx == 0:
            ax.set_ylabel('Query', fontsize=9)
        if layer_idx == n_layers - 1:
            ax.set_xlabel('Key', fontsize=9)

plt.suptitle('All Attention Heads (Layer × Head)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Attention pattern analysis: classify each head's behaviour
# ============================================================

print("="*70)
print("ATTENTION HEAD PATTERN ANALYSIS")
print("="*70)

for layer_idx in range(n_layers):
    for head_idx in range(n_heads):
        w = attn_weights[layer_idx][head_idx].numpy()
        seq_len = w.shape[0]
        
        # Measure diagonal dominance (self-attention)
        diag_weight = np.mean([w[i, i] for i in range(seq_len)])
        
        # Measure first-token attention
        first_token_weight = np.mean(w[:, 0])
        
        # Measure previous-token attention (local)
        prev_weight = np.mean([w[i, max(0, i-1)] for i in range(seq_len)])
        
        # Measure entropy (how spread out the attention is)
        # High entropy = uniform attention, low entropy = focused
        entropy = -np.mean(np.sum(w * np.log(w + 1e-10), axis=-1))
        max_entropy = np.log(seq_len)  # uniform distribution
        
        # Classify pattern
        pattern = ""
        if diag_weight > 0.4:
            pattern = "Self-focused (attends mainly to self)"
        elif first_token_weight > 0.4:
            pattern = "Position-anchored (attends to first token)"
        elif prev_weight > 0.4:
            pattern = "Local (attends to previous token)"
        elif entropy / max_entropy > 0.85:
            pattern = "Broad (roughly uniform attention)"
        else:
            pattern = "Mixed (no dominant pattern)"
        
        print(f"\n  Layer {layer_idx}, Head {head_idx}: {pattern}")
        print(f"    Diagonal (self):  {diag_weight:.3f}")
        print(f"    First token:      {first_token_weight:.3f}")
        print(f"    Previous token:   {prev_weight:.3f}")
        print(f"    Entropy ratio:    {entropy/max_entropy:.3f} (1.0 = uniform)")

## 4. Attention Rollout

Attention rollout (Abnar & Zuidema, 2020) estimates how much each **input** token contributes to each **output** position, by propagating attention through all layers.

The idea:
1. At each layer, account for the residual connection: $\hat{A} = 0.5 \cdot A + 0.5 \cdot I$
2. Multiply these matrices across layers: $R = \hat{A}_1 \cdot \hat{A}_2 \cdot \ldots \cdot \hat{A}_L$

This gives a single matrix showing the effective attention from input to output positions.

In [ ]:
# ============================================================
# Compute and visualise attention rollout
# ============================================================

rollout = compute_attention_rollout(attn_weights)

print(f"Rollout matrix shape: {rollout.shape}  — (seq, seq)")
print(f"Each row shows how much each input token contributes to that output position.")

plot_attention_rollout(
    rollout,
    tokens=tokens,
    title=f"Attention Rollout ({n_layers} layers combined)",
    figsize=(9, 8),
    annot=len(tokens) <= 10
)

In [ ]:
# ============================================================
# Compare: Layer 0 attention vs Rollout (accumulated)
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Layer 0, averaged over heads
avg_layer0 = attn_weights[0].mean(dim=0).numpy()  # (seq, seq)
plot_attention_heatmap(
    avg_layer0, tokens,
    title="Layer 0 Average Attention (single layer)",
    ax=ax1, annot=len(tokens) <= 10
)

# Rollout (all layers combined)
sns.heatmap(
    rollout, ax=ax2,
    xticklabels=tokens, yticklabels=tokens,
    cmap='viridis', vmin=0,
    annot=len(tokens) <= 10, fmt='.2f',
    linewidths=0.3, linecolor='white',
    square=True
)
ax2.set_title('Attention Rollout (all layers)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Input token')
ax2.set_ylabel('Output position')
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('Single Layer vs Accumulated Attention', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nThe rollout shows how attention ACCUMULATES across layers.")
print("A token might not directly attend to another in layer 0,")
print("but through intermediate tokens across multiple layers,")
print("information can still flow from distant positions.")

## 5. Interactive: Visualise Your Own Text

Change the text below and re-run the cell to see attention patterns for any input.

In [ ]:
# ============================================================
# 🔧 CHANGE THIS TEXT and re-run the cell!
# ============================================================

your_text = "A transformer is a neural network"

# ============================================================
# Visualise
# ============================================================

your_attn, your_tokens = get_attention_for_text(model, your_text, enc)

print(f"Text: \"{your_text}\"")
print(f"Tokens ({len(your_tokens)}): {your_tokens}")
print()

# Show all heads for each layer
for layer_idx in range(len(your_attn)):
    plot_multi_head_attention(
        your_attn[layer_idx],
        tokens=your_tokens,
        title=f"Layer {layer_idx}",
        annot=len(your_tokens) <= 12
    )

# Rollout
your_rollout = compute_attention_rollout(your_attn)
plot_attention_rollout(
    your_rollout,
    tokens=your_tokens,
    title="Attention Rollout",
    annot=len(your_tokens) <= 12
)

In [ ]:
# ============================================================
# Bar chart: For a specific output token, which input tokens
# contributed most (according to rollout)?
# ============================================================

# Pick the last token's perspective (what does the final output attend to?)
query_pos = -1  # last token
query_token = your_tokens[query_pos]
contributions = your_rollout[query_pos]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(your_tokens)), contributions, color='teal', alpha=0.8)
ax.set_xticks(range(len(your_tokens)))
ax.set_xticklabels(your_tokens, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Rollout Attribution', fontsize=11)
ax.set_title(f'Input token contributions to output "{query_token}" (pos {len(your_tokens)+query_pos})',
             fontsize=12, fontweight='bold')

# Annotate top values
for bar, v in zip(bars, contributions):
    if v > 0.05:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.2f}', ha='center', fontsize=8)

ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

## 6. Interpreting Attention Patterns

---

### Common patterns you might see:

| Pattern | What it looks like | What it might mean |
|---------|-------------------|-------------------|
| **Strong diagonal** | Bright diagonal line | Each token attends mostly to itself |
| **First-column bright** | Leftmost column highlighted | Tokens attend to the first token (like a "default") |
| **Sub-diagonal** | Bright line just below diagonal | Each token attends to the previous token (bigram pattern) |
| **Uniform rows** | Similar colour across row | Broad, unfocused attention (gathering general context) |
| **Sparse bright spots** | Few bright cells, rest dark | Very focused attention on specific tokens |
| **Lower triangle only** | Upper triangle is dark | Causal mask working correctly |

### Caveats

- **Our model is tiny** — patterns may not be as clean as in large pre-trained models
- **Attention ≠ explanation** — high attention doesn't necessarily mean that token was "important" for the prediction
- **Residual connections** complicate interpretation — information can bypass attention entirely
- **Rollout** is a better approximation of information flow than raw attention weights

### Further Reading

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — the original transformer paper
- [BertViz](https://github.com/jessevig/bertviz) — interactive attention visualisation for large models
- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) — Anthropic's deep dive into attention patterns

---

### 🎉 Congratulations!

You've built a transformer from scratch, trained it, and visualised its internal workings. The same architecture (just scaled up) powers GPT-4, Claude, and every modern large language model.

---